# 7 Wonders: symulacja i trening ONNX

Ten notebook uruchamia symulację w C#, zbiera plik wynikowy z logami ruchów i trenuje hierarchiczny model policy/value na faktycznym wektorze stanu.

Przepływ:
1. `GameConsole` eksportuje dane z `MoveLog.State`, `MoveLog.ActionMask` i `MoveLog.ActionIndex`.
2. Notebook wczytuje najnowszy plik `training_*.json`.
3. Model PyTorch uczy się i eksportuje `policy_network.onnx`.

In [25]:
EPOCHS = 50
GAMES_TRAIN = 200

In [26]:
from pathlib import Path
import importlib
import subprocess
import sys

repo_root = Path(r"c:/Users/kubeu/Kuba-dokumenty/Magisterka/7 Wonders")
game_console = repo_root / "GameConsole" / "GameConsole.csproj"
results_dir = repo_root / "GameConsole" / "Results"
encoding_dir = repo_root / "GameAI" / "Encoding"

sys.path.append(str(encoding_dir))

import game_training_pipeline as gtp
gtp = importlib.reload(gtp)

ActionSpace = gtp.ActionSpace
GameDataset = gtp.GameDataset
HierarchicalPolicyNetwork = gtp.HierarchicalPolicyNetwork
train_epoch = gtp.train_epoch
evaluate = gtp.evaluate

print("Repo root:", repo_root)
print("State vector size:", ActionSpace.STATE_VECTOR_SIZE)
print("Primary action size:", ActionSpace.TOTAL_PRIMARY_ACTIONS)

Repo root: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders
State vector size: 1903
Primary action size: 120


In [27]:
import torch
print(f"Czy CUDA działa? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Wykryta karta: {torch.cuda.get_device_name(0)}")
    print(f"Wersja CUDA w Torch: {torch.version.cuda}") # type: ignore

Czy CUDA działa? True
Wykryta karta: NVIDIA GeForce RTX 4050 Laptop GPU
Wersja CUDA w Torch: 12.4


In [28]:
def run_simulation(seed: int = 12345, games: int = 20, agent1: str = "heuristic-personal", agent2: str = "mcts"):
    results_dir.mkdir(parents=True, exist_ok=True)
    command = [
        "dotnet", "run",
        "--project", str(game_console),
        "--",
        "export-data",
        "--seed", str(seed),
        "--games", str(games),
        "--agent1", agent1,
        "--agent2", agent2,
    ]
    subprocess.run(command, cwd=repo_root, check=True)

run_simulation(seed=1, games=GAMES_TRAIN, agent1="heuristic-personal", agent2="onnx2")
print("Simulation finished.")

Simulation finished.


In [29]:
from torch.utils.data import DataLoader, random_split
import torch

training_files = sorted(results_dir.glob("training_*.json"))
if not training_files:
    raise FileNotFoundError(f"No training_*.json files found in {results_dir}")

latest_file = max(training_files, key=lambda p: p.stat().st_mtime)
print("Using dataset:", latest_file)

dataset = GameDataset(str(latest_file), normalize=True, validate_shapes=True)
train_size = max(1, int(len(dataset) * 0.9))
val_size = max(1, len(dataset) - train_size)
if train_size + val_size > len(dataset):
    val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) if len(val_dataset) > 0 else None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HierarchicalPolicyNetwork(state_dim=ActionSpace.STATE_VECTOR_SIZE, hidden_dim=256, dropout=0.1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(EPOCHS):
    train_stats = train_epoch(model, train_loader, optimizer, device)
    if val_loader is not None:
        val_stats = evaluate(model, val_loader, device)
        print(f"epoch={epoch + 1} train={train_stats['total_loss']:.4f} val={val_stats['total_loss']:.4f}")
    else:
        print(f"epoch={epoch + 1} train={train_stats['total_loss']:.4f}")

2026-05-07 01:43:06,258 - game_training_pipeline - INFO - Loading dataset from c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameConsole\Results\training_20260507_014303_heuristic-personal_vs_onnx2_200_games.json


Using dataset: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameConsole\Results\training_20260507_014303_heuristic-personal_vs_onnx2_200_games.json


2026-05-07 01:43:10,522 - game_training_pipeline - INFO - Loaded 11443 valid samples
2026-05-07 01:43:10,654 - game_training_pipeline - INFO - State normalized: mean=0.0473, std=0.1081


epoch=1 train=1.4347 val=1.2209
epoch=2 train=1.0453 val=1.1300
epoch=3 train=0.7755 val=1.1591
epoch=4 train=0.5128 val=1.4135
epoch=5 train=0.3132 val=1.8896
epoch=6 train=0.2026 val=2.1267
epoch=7 train=0.1454 val=2.7746
epoch=8 train=0.1216 val=2.9413
epoch=9 train=0.1013 val=3.4152
epoch=10 train=0.0922 val=3.4765
epoch=11 train=0.0858 val=3.7963
epoch=12 train=0.0855 val=3.8840
epoch=13 train=0.0903 val=3.9064
epoch=14 train=0.0679 val=4.2502
epoch=15 train=0.0830 val=4.4733
epoch=16 train=0.0858 val=4.3361
epoch=17 train=0.0921 val=4.4933
epoch=18 train=0.0867 val=4.6595
epoch=19 train=0.0651 val=5.0373
epoch=20 train=0.0696 val=5.3210
epoch=21 train=0.0728 val=5.6286
epoch=22 train=0.0679 val=5.5021
epoch=23 train=0.0616 val=5.9235
epoch=24 train=0.0720 val=5.7435
epoch=25 train=0.0628 val=5.9497
epoch=26 train=0.0654 val=5.7579
epoch=27 train=0.0709 val=6.1526
epoch=28 train=0.0651 val=6.3925
epoch=29 train=0.0617 val=6.6666
epoch=30 train=0.0737 val=6.7698
epoch=31 train=0.05

In [30]:
onnx_path = encoding_dir / f"onnx_models/policy_network_{EPOCHS}_{GAMES_TRAIN}.onnx"
model.onnx_export(str(onnx_path), validate=True)
print("Exported:", onnx_path)

batch = next(iter(train_loader))
state = batch['state'].to(device)
action_mask = batch['action_mask'].to(device)
outputs = model(state, action_mask=action_mask)
print("policy shape:", outputs['policy_masked_logits'].shape)
print("value shape:", outputs['value'].shape)

2026-05-07 01:45:26,243 - game_training_pipeline - INFO - Exporting model to ONNX format: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_200.onnx
2026-05-07 01:45:26,314 - game_training_pipeline - INFO - ✓ Model exported successfully: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_200.onnx
2026-05-07 01:45:26,315 - game_training_pipeline - INFO - Validating ONNX model: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_200.onnx
2026-05-07 01:45:26,467 - game_training_pipeline - INFO -   Inputs: ['state', 'action_mask']
2026-05-07 01:45:26,469 - game_training_pipeline - INFO -   Outputs: ['policy_masked_logits', 'value']
2026-05-07 01:45:26,469 - game_training_pipeline - INFO - ✓ ONNX model validation passed


Exported: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_200.onnx
policy shape: torch.Size([32, 120])
value shape: torch.Size([32, 1])
